# AMR Drug Discovery with AI Agents

**Full autonomous workflow: Target selection → Molecule design → Docking → Resistance prediction**

This notebook demonstrates:
1. AI agent meeting to decide targets
2. Dynamic structure download
3. LLM-driven molecule generation
4. Molecular docking
5. Resistance prediction
6. World model state tracking

---

## Cell 1: API Configuration

In [ ]:
import os
import getpass

if not os.environ.get('OPENAI_API_KEY'):
    print("OpenAI API key required for AI agents")
    print("Get your key from: https://platform.openai.com/api-keys\n")
    api_key = getpass.getpass("Enter OpenAI API key: ")
    os.environ['OPENAI_API_KEY'] = api_key
    print("✓ API key configured")
else:
    print("✓ API key already set")

from openai import OpenAI
client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])
print("✓ OpenAI client initialized")

## Cell 2: Import Modules

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))

from src.bioknowledge import ResistanceDatabase, PathogenDatabase, TargetProteinDB, CARDDownloader
from src.docking import StructureDownloader, VinaWrapper, ReceptorPrep
from src.genomics import ResistanceGenomicsAnalyzer
from src.microbiology import MICAnalyzer, StrainManager
from src.world_model import WorldStateTracker, KnowledgeGraph, KosmosEngine
from src.agents.amr_agents import MicrobiologyAgent, GenomicsAgent, CheminformaticsAgent, ResistanceCriticAgent
from src.molecule_design import MoleculeGenerator, MoleculeOptimizer
from src.admet import DrugLikenessCalculator
from src.resistance import ResistancePredictor
from src.core import setup_logger

logger = setup_logger('khukuri')
print("✓ All modules imported")

## Cell 3: Initialize System

In [ ]:
print("Initializing Khukuri AMR Discovery System...\n")

# Download/update databases
card = CARDDownloader()
card.download_card()
card.update_resistance_db_file()

# Initialize databases
resistance_db = ResistanceDatabase()
pathogen_db = PathogenDatabase()
target_db = TargetProteinDB()

# Initialize tools
structure_downloader = StructureDownloader()
genomics_analyzer = ResistanceGenomicsAnalyzer()
mic_analyzer = MICAnalyzer()
strain_manager = StrainManager()

# Initialize world model
world_state = WorldStateTracker()
knowledge_graph = KnowledgeGraph()
kosmos = KosmosEngine(world_state, knowledge_graph)

# Initialize AI agents
micro_agent = MicrobiologyAgent(client)
genomics_agent = GenomicsAgent(client)
chem_agent = CheminformaticsAgent(client)
critic_agent = ResistanceCriticAgent(client)

print("✓ System initialized")
print(f"  • Resistance genes: {len(resistance_db.genes)}")
print(f"  • Pathogens: {len(pathogen_db.pathogens)}")
print(f"  • AI agents: 4 active")

## Cell 4: Define Target Pathogen

In [ ]:
# Select pathogen
PATHOGEN = "Mycobacterium tuberculosis"
PRIORITY = "critical"

print(f"Target Pathogen: {PATHOGEN}")
print(f"WHO Priority: {PRIORITY}\n")

# Get pathogen info
pathogen_info = pathogen_db.get_pathogen_info(PATHOGEN)

print("Pathogen Information:")
print(f"  • WHO Priority: {pathogen_info['who_priority']}")
print(f"  • Known targets: {len(pathogen_info['targets'])}")
print(f"  • Resistance mechanisms: {len(pathogen_info['resistance_mechanisms'])}")

print("\nKnown drug targets:")
for target in pathogen_info['targets'][:5]:
    print(f"  • {target}")

# Get resistance genes
resistance_genes = resistance_db.get_genes_by_organism(PATHOGEN)
print(f"\nResistance genes: {len(resistance_genes)}")
for gene in resistance_genes:
    info = resistance_db.query_gene(gene)
    print(f"  • {gene}: {info['type']} resistance")

## Cell 5: AI Agent Meeting - Target Selection

In [ ]:
print("="*60)
print("AI AGENT MEETING: TARGET SELECTION")
print("="*60 + "\n")

# Microbiology Agent analysis
print("[Microbiology Agent] Analyzing pathogen...")
micro_analysis = micro_agent.analyze_pathogen(
    PATHOGEN,
    resistance_genes,
    pathogen_info
)
print(f"Analysis: {micro_analysis['summary']}")
print(f"Recommended focus: {', '.join(micro_analysis['recommended_targets'][:3])}\n")

# Genomics Agent analysis
print("[Genomics Agent] Analyzing resistance mutations...")
genomics_analysis = genomics_agent.analyze_resistance_profile(
    PATHOGEN,
    resistance_genes
)
print(f"Analysis: {genomics_analysis['summary']}")
print(f"High-risk targets: {', '.join(genomics_analysis['high_risk_targets'])}\n")

# Cheminformatics Agent recommendation
print("[Cheminformatics Agent] Evaluating druggability...")
chem_analysis = chem_agent.evaluate_targets(
    pathogen_info['targets'],
    target_db
)
print(f"Analysis: {chem_analysis['summary']}")
print(f"Top druggable targets: {', '.join(chem_analysis['top_targets'][:3])}\n")

# Resistance Critic evaluation
print("[Resistance Critic] Evaluating strategy...")
critic_eval = critic_agent.evaluate_strategy(
    micro_analysis['recommended_targets'],
    genomics_analysis['high_risk_targets'],
    chem_analysis['top_targets']
)
print(f"Evaluation: {critic_eval['assessment']}")
print(f"Final recommendation: {', '.join(critic_eval['final_targets'])}\n")

# Store decision in world model
SELECTED_TARGETS = critic_eval['final_targets']
for target in SELECTED_TARGETS:
    world_state.update_target(target, {'selected': True, 'pathogen': PATHOGEN})
    knowledge_graph.add_target(target, {'druggability': 0.8})

print("="*60)
print(f"DECISION: Selected {len(SELECTED_TARGETS)} targets")
print("="*60)